# Übung: Zweidimensionales Ising-Modell

In dieser Übung untersuchen wir den Phasenübergang des zweidimensionalen Ising-Modells mit lokalen Metropolis-Updates. 

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from numba import njit

notebook_name = 'IsingExercise_solution'
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.git').exists()),
    Path.cwd(),
)
output_dir = project_root / 'data' / '02_statphys' / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)

J = 1.0
Tc = 2.0 * J / np.log(1.0 + np.sqrt(2.0))
rng = np.random.default_rng(2627)
print(f'Tc = {Tc:.6f}')
print(f'Output directory: {output_dir}')

## 1. Lokales Metropolis-Update

Auf jedem Platz eines quadratischen $L\times L$-Gitters sitzt ein Spin $s_{ij}\in\{-1,+1\}$. Mit periodischen Randbedingungen lautet die Energie

$$E=-J\sum_{\langle ij,kl\rangle}s_{ij}s_{kl},$$

wobei $(i,j)$ mit $i,j\in\{0,...,L-1\}$ die zweidimensionalen Koordinaten des Gitterplatzes bezeichnet. 

Wird ein einzelner Spin umgedreht, ändert sich die Energie um einen Wert $\Delta E$. Implementieren Sie diese Energie `delta_energy` und die Funktion  `metropolis_sweep`, die zukzessive an $L^2$ zufällig ausgewualten Plätzen einen Spin-Flip versucht. 

In [ ]:
@njit
def delta_energy(state, i, j, J=1.0):
    L = state.shape[0]
    neighbor_sum = (state[(i - 1) % L, j] + state[(i + 1) % L, j]
                    + state[i, (j - 1) % L] + state[i, (j + 1) % L])
    return 2.0 * J * state[i, j] * neighbor_sum

@njit
def metropolis_sweep(state, beta, J=1.0):
    L = state.shape[0]
    accepted = 0
    for _ in range(L * L):
        i = np.random.randint(0, L)
        j = np.random.randint(0, L)
        dE = delta_energy(state, i, j, J)
        if dE <= 0.0 or np.random.random() < np.exp(-beta * dE):
            state[i, j] *= -1
            accepted += 1
    return accepted / (L * L)

@njit
def total_energy(state, J=1.0):
    # Jede Bindung wird nur nach rechts und nach unten gezählt.
    L = state.shape[0]
    energy = 0.0
    for i in range(L):
        for j in range(L):
            energy -= J * state[i, j] * (state[(i + 1) % L, j] + state[i, (j + 1) % L])
    return energy

def random_state(L, rng):
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=(L, L))

## 2. Konfigurationen in drei Temperaturbereichen

Erzeugen Sie Zustände bei $T/T_c=0.7$, $1.0$ und $1.4$. 

In [ ]:
L_snapshot = 40
temperature_ratios = [0.7, 1.0, 1.4]
n_warm_snapshot = 3000

states = []
for ratio in temperature_ratios:
    state = random_state(L_snapshot, rng)
    beta = 1.0 / (ratio * Tc)
    for _ in range(n_warm_snapshot):
        metropolis_sweep(state, beta, J)
    states.append(state.copy())

fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, state, ratio in zip(axes, states, temperature_ratios):
    ax.imshow(state, cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(fr'$T/T_c={ratio}$')
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig(output_dir / 'ising_configurations.pdf', bbox_inches='tight')
plt.show()

## 3. Temperaturabhängige Observablen

Nach dem Warmup messen wir nach jedem Sweep die Gesamtenergie $E$ und die Gesamtmagnetisierung $M=\sum_i s_i$. Für ein endliches System verschwindet $\langle M\rangle$ langfristig wegen der Symmetrie zwischen den beiden Magnetisierungsrichtungen. Verwenden Sie deshalb $\langle|M|\rangle$.

Berechnen Sie pro Gitterplatz

$$e=\frac{\langle E\rangle}{L^2},\qquad m=\frac{\langle|M|\rangle}{L^2},$$

und die Suszeptibilität

$$\chi=\frac{\beta}{L^2}\left(\langle M^2\rangle-\langle|M|\rangle^2\right).$$

Simulieren Sie mehrere kleine Gittergrößen. Bewahren Sie die vollständigen Magnetisierungsreihen auf; sie werden in den nächsten Abschnitten erneut verwendet.

In [ ]:
@njit
def measure_chain(state, beta, n_warm, n_steps, J=1.0):
    for _ in range(n_warm):
        metropolis_sweep(state, beta, J)
    energies = np.empty(n_steps)
    magnetizations = np.empty(n_steps)
    for step in range(n_steps):
        metropolis_sweep(state, beta, J)
        energies[step] = total_energy(state, J)
        magnetizations[step] = np.sum(state)
    return energies, magnetizations

sizes = [8, 12, 16, 20]
temperatures = Tc * np.linspace(0.90, 1.10, 11)
n_warm = 5000
# numer of steps N=2**pmax to allow for a simple binning analysis
pmax=18
n_steps = 2**pmax
scan_data = {}

# this simulation taked ~ 1 min
for L in sizes:
    for T in temperatures:
        state = random_state(L, rng)
        scan_data[(L, T)] = measure_chain(state, 1.0 / T, n_warm, n_steps, J)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for L_plot in sizes:
    energy_density = []
    magnetization_density = []
    susceptibility = []
    for T in temperatures:
        energies, magnetizations = scan_data[(L_plot, T)]
        energy_density.append(energies.mean() / L_plot**2)
        magnetization_density.append(np.mean(np.abs(magnetizations)) / L_plot**2)
        susceptibility.append((np.mean(magnetizations**2)
                              - np.mean(np.abs(magnetizations))**2) / (T * L_plot**2))

    axes[0].plot(temperatures / Tc, energy_density, 'o-',label=f'L={L_plot}')
    axes[0].set(xlabel='$T/T_c$', ylabel=r'$\langle E\rangle/L^2$')
    axes[1].plot(temperatures / Tc, magnetization_density, 'o-',label=f'L={L_plot}')
    axes[1].set(xlabel='$T/T_c$', ylabel=r'$\langle|M|\rangle/L^2$')
    axes[2].plot(temperatures / Tc, susceptibility, 'o-')
    axes[2].set(xlabel='$T/T_c$', ylabel=r'$\chi$',label=f'L={L_plot}')
for ax in axes:
    ax.axvline(1.0, color='black', linestyle='--', linewidth=1)
axes[0].legend()
fig.tight_layout()
fig.savefig(output_dir / 'ising_observables.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Temperaturabhängige Fehlerbalken aus einer Binning-Analyse
# Wir verwenden |M|/L^2 für das größte simulierte L.
def binning_error(data, block_size):
    n_blocks = len(data) // block_size
    trimmed = data[:n_blocks * block_size]
    block_means = trimmed.reshape(n_blocks, block_size).mean(axis=1)
    return np.sqrt(block_means.var() / n_blocks)

L_error = sizes[-1]
min_blocks = 32
magnetization_means = []
plateau_errors = []

# do the binning analysis for the tempoerature in the middle:
T=temperatures[len(temperatures)//2]
series = np.abs(scan_data[(L_error, T)][1]) / L_error**2
largest_block = max(1, len(series) // min_blocks)
max_power = int(np.floor(np.log2(largest_block)))
block_sizes = 2 ** np.arange(max_power + 1)
errors = np.array([binning_error(series, size) for size in block_sizes])

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(block_sizes,errors,'-o')
ax.set_xscale('log')
ax.set_ylabel(r'Error (bin size $M$): $\sqrt{var(O_M)/N_{bins}}$')
ax.set_xlabel(r'$M$')
fig.savefig(output_dir / 'magnetization_binning_analyis.pdf', bbox_inches='tight')
plt.show()

# based on this plot, I will use an average of the last three values as 1-sigma error

In [ ]:
# do the binning for all temperatures:
for T in temperatures:
    series = np.abs(scan_data[(L_error, T)][1]) / L_error**2
    largest_block = max(1, len(series) // min_blocks)
    max_power = int(np.floor(np.log2(largest_block)))
    block_sizes = 2 ** np.arange(max_power + 1)
    errors = np.array([binning_error(series, size) for size in block_sizes])
    magnetization_means.append(series.mean())
    # Median der drei größten zulässigen Blockgrößen
    plateau_errors.append(np.median(errors[-3:]))

magnetization_means = np.asarray(magnetization_means)
plateau_errors = np.asarray(plateau_errors)
three_sigma_errors = 3.0 * plateau_errors

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(temperatures / Tc, magnetization_means, yerr=three_sigma_errors,
            fmt='o-', capsize=3, label=fr'$L={L_error}$')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='$T_c$')
ax.set(xlabel='$T/T_c$', ylabel=r'$\langle|M|\rangle/L^2$',
       title=r'Magnetisierung mit $3\sigma$-Binning-Fehlern')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'magnetization_binning_error.pdf', bbox_inches='tight')
plt.show()

print(' T/Tc       <|M|>/L^2       3-sigma-Fehler')
for T, mean, error in zip(temperatures, magnetization_means, three_sigma_errors):
    print(f'{T/Tc:5.2f}       {mean:10.6f}       {error:10.6f}')

## 4. Binder-Kumulante und Finite-Size-Effekt

Die dimensionslose Binder-Kumulante ist

$$U_4(T,L)=1-\frac{\langle M^4\rangle}{3\langle M^2\rangle^2}.$$

Nahe dem kritischen Punkt werden die Kurven für verschiedene $L$ ungefähr unabhängig von $L$ und kreuzen sich. Berechnen Sie $U_4$ aus den bereits erzeugten Magnetisierungsreihen und vergleichen Sie die Kreuzungsregion mit dem exakten $T_c$.

Überlegemn Sie ggf, warum es zur Kreuzung bei Tc kommt? (Diskussion an der Tafel)


In [ ]:
def binder_cumulant(magnetizations):
    second_moment = np.mean(magnetizations**2)
    fourth_moment = np.mean(magnetizations**4)
    return 1.0 - fourth_moment / (3.0 * second_moment**2)

fig, ax = plt.subplots(figsize=(6, 4))
for L in sizes:
    binder_values = [binder_cumulant(scan_data[(L, T)][1]) for T in temperatures]
    ax.plot(temperatures / Tc, binder_values, 'o-', label=f'$L={L}$')
ax.axvline(1.0, color='black', linestyle='--', label='exaktes $T_c$')
ax.set(xlabel='$T/T_c$', ylabel='$U_4$', title='Binder-Kumulante')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'binder_cumulant.pdf', bbox_inches='tight')
plt.show()

## 5. Größere Wechselwirkungsreichweite und Mean-Field-Grenze

Nun vergrößern wir die Nachbarschaft eines Spins, wobei jede Bindung weiterhin dieselbe Stärke $J$ besitzt:

- $z=4$: die vier nächsten axialen Nachbarn (NN),
- $z=8$: zusätzlich die vier diagonalen Nachbarn (NN+NNN),
- $z=12$: zusätzlich die vier axialen Nachbarn im Abstand zwei.

Welche Temperatur $T_c^{\mathrm{MF}}(z)$ erhlatne Sie aus der Mean field Theorie?

Verallgemeinern Sie das lokale Update auf eine Liste von Nachbarvektoren. Bestimmen Sie für $z=4,8,12$ jeweils Binder-Kurven für mehrere Gittergrößen und extrahieren Sie aus der Kreuzung der beiden größten Systeme einen numerischen Schätzwert für $T_c(z)$.

Vergleichen Sie $T_c(z)$ sowohl mit $T_c^{\mathrm{MF}}(z)$. Was beobachten Sie? 

In [ ]:
neighbor_sets = {
    4: np.array([(-1, 0), (1, 0), (0, -1), (0, 1)], dtype=np.int64),
    8: np.array([(-1, 0), (1, 0), (0, -1), (0, 1),
                 (-1, -1), (-1, 1), (1, -1), (1, 1)], dtype=np.int64),
    12: np.array([(-1, 0), (1, 0), (0, -1), (0, 1),
                  (-1, -1), (-1, 1), (1, -1), (1, 1),
                  (-2, 0), (2, 0), (0, -2), (0, 2)], dtype=np.int64),
}

@njit
def delta_energy_range(state, i, j, neighbors, J=1.0):
    L = state.shape[0]
    neighbor_sum = 0
    for k in range(neighbors.shape[0]):
        di, dj = neighbors[k, 0], neighbors[k, 1]
        neighbor_sum += state[(i + di) % L, (j + dj) % L]
    return 2.0 * J * state[i, j] * neighbor_sum

@njit
def metropolis_sweep_range(state, beta, neighbors, J=1.0):
    L = state.shape[0]
    for _ in range(L * L):
        i = np.random.randint(0, L)
        j = np.random.randint(0, L)
        dE = delta_energy_range(state, i, j, neighbors, J)
        if dE <= 0.0 or np.random.random() < np.exp(-beta * dE):
            state[i, j] *= -1

@njit
def measure_magnetization_range(state, beta, neighbors, n_warm, n_steps, J=1.0):
    for _ in range(n_warm):
        metropolis_sweep_range(state, beta, neighbors, J)
    magnetizations = np.empty(n_steps)
    for step in range(n_steps):
        metropolis_sweep_range(state, beta, neighbors, J)
        magnetizations[step] = np.sum(state)
    return magnetizations

binder_by_z = {}

range_temperature_ratios = np.linspace(0.50, 0.85, 11)
n_warm_range = 4000
n_steps_range = 100_000
for z in [4, 8, 12]:
    range_temperatures = z * J * range_temperature_ratios
    curves = {}
    for L in sizes:
        values = []
        for T in range_temperatures:
            state = random_state(L, rng)
            m_values = measure_magnetization_range(
                state, 1.0 / T, neighbor_sets[z], n_warm_range, n_steps_range, J
            )
            values.append(binder_cumulant(m_values))
        curves[L] = np.asarray(values)
    binder_by_z[z] = (range_temperatures, curves)


In [ ]:
# plot the binder ratios:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, z in zip(axes, [4, 8, 12]):
    temperature_values, curves = binder_by_z[z]
    for L in sizes:
        ax.plot(temperature_values / (z * J), curves[L], 'o-', label=f'$L={L}$')
    ax.set(xlabel=r'$T/(zJ)$', title=f'$z={z}$', xlim=(0.45, 1.03))
axes[0].set_ylabel('$U_4$')
axes[-1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / 'binder_interaction_range.pdf', bbox_inches='tight')
plt.show()


**Diskussion:** Rückt $T_c^{\mathrm{MF}}(z)$ für größere $z$ näher an den Mean-Field-Wert? Was sagt das über die kritischen Exponenten des Modells (die wir hoer nicht messen)?

## 6. Kritisches Slowing-down

Nahe $T_c$ entstehen große korrelierte Domänen. Lokale Updates verändern sie nur langsam. Wir quantifizieren dies mit der normierten Autokorrelation von $O=|M|/L^2$:

$$\rho_O(t)=\frac{\langle(O_i-\langle O\rangle)(O_{i+t}-\langle O\rangle)\rangle}{\operatorname{Var}(O)}.$$

Bestimmen Sie als robuste, einfache Zeitskala den ersten Abstand $\tau_{1/e}$, bei dem $\rho_O(t)<e^{-1}$ gilt. Simulieren Sie dazu bei $T_c$ mehrere Gittergrößen, einschließlich $L=20$. Plotten Sie die Autokorrelationen und anschließend $\tau_{1/e}$ gegen $L$ auf logarithmischen Achsen.

Für lokale Dynamik erwartet man näherungsweise $\tau\sim L^z$ mit $z\approx2$. Eine genaue Bestimmung von $z$ ist hier nicht das Ziel. Erklären Sie stattdessen, weshalb das Slowing-down die Binder-Analyse für größere Systeme zunehmend teuer macht.

In [ ]:
def autocorrelation(data, max_lag):
    centered = np.asarray(data, dtype=float) - np.mean(data)
    n = len(centered)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(centered, n=n_fft)
    corr = np.fft.irfft(spectrum * np.conjugate(spectrum), n=n_fft)[:n]
    corr /= np.arange(n, 0, -1)
    max_lag = min(max_lag, n - 1)
    return corr[:max_lag + 1] / corr[0]

def correlation_time_1e(rho):
    crossings = np.flatnonzero(rho < np.exp(-1))
    return float(crossings[0]) if len(crossings) else np.nan

slowing_sizes = [8, 12, 16, 20]
n_warm_slowing = 10_000
n_steps_slowing = 50_000
max_lag = 200

autocorrelations = {}
correlation_times = []
for L in slowing_sizes:
    state = random_state(L, rng)
    _, magnetizations = measure_chain(
        state, 1.0 / Tc, n_warm_slowing, n_steps_slowing, J
    )
    observable = np.abs(magnetizations) / L**2
    rho = autocorrelation(observable, max_lag)
    autocorrelations[L] = rho
    correlation_times.append(correlation_time_1e(rho))

fig, ax = plt.subplots(figsize=(6, 4))
for L, rho in autocorrelations.items():
    ax.plot(np.arange(len(rho)), rho, label=f'$L={L}$')
ax.axhline(np.exp(-1), color='black', linestyle='--', label='$e^{-1}$')
ax.set(xlabel='Abstand $t$ [Sweeps]', ylabel=r'$\rho_{|M|}(t)$',
       title='Autokorrelation bei $T_c$', xlim=(0, max_lag), ylim=(-0.1, 1.02))
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'critical_autocorrelation.pdf', bbox_inches='tight')
plt.show()

correlation_times = np.asarray(correlation_times)
sizes_array = np.asarray(slowing_sizes, dtype=float)
reference = correlation_times[0] * (sizes_array / sizes_array[0])**2
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(sizes_array, correlation_times, 'o-', label=r'$\tau_{1/e}$')
ax.loglog(sizes_array, reference, '--', label=r'Referenz $\sim L^2$')
ax.set(xlabel='$L$', ylabel=r'$\tau_{1/e}$ [Sweeps]',
       title='Kritisches Slowing-down')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'critical_slowing_down.pdf', bbox_inches='tight')
plt.show()

for L, tau in zip(slowing_sizes, correlation_times):
    print(f'L={L:2d}: tau_1/e = {tau:.0f} Sweeps')

### Kurze Antworten

- **Binder-Kreuzung:** $U_4$ ist bei $T_c$ im führenden Skalengesetz unabhängig von $L$; Statistik und Korrekturen verschieben die Kreuzungen leicht.
- **Mean Field:** $T_c^{\mathrm{MF}}=zJ$. Numerisch nähert sich $T_c/(zJ)$ der Eins, die endliche Reichweite behält aber die 2D-Ising-Exponenten.
- **Slowing-down:** $\tau$ wächst ungefähr wie $L^2$; dadurch sinkt die Zahl effektiv unabhängiger Messungen.